[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/07_logistic_regression/93_assignment_softmax_regression.ipynb)

# Assignment 93 — Softmax Regression: từ chữ số viết tay đến Fashion-MNIST

> Khoá AIVN đã lên kế hoạch bài tập *"Softmax Regression cho bài toán MNIST Fashion"* ở Tuần 5 nhưng chưa có nội dung. Assignment này hiện thực hoá ý tưởng đó: cài đặt và kiểm tra trên bộ `digits` (offline, 10 lớp, mục 1–5), rồi áp dụng cho **Fashion-MNIST** thật (mục 6). Học trước bài [06](../../../notebooks/07_logistic_regression/06_softmax_regression.ipynb).

### Hướng dẫn làm bài
- Quy ước: **mỗi hàng là một mẫu**; $X$: $(m, n)$, $W$: $(n, C)$, $\mathbf b$: $(C,)$, $Y$ one-hot: $(m, C)$.
- Vector hoá hoàn toàn; viết code trong `### START CODE HERE ###` … `### END CODE HERE ###`.

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

from ml_course.checks import check_close, check_greater, check_shape, check_true
from ml_course.datasets import load_digits_data
from ml_course.utils import numerical_gradient, relative_error

X, y, images = load_digits_data()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
C = 10
print(X_train.shape, X_val.shape)

## 1. One-hot encoding

**Bài tập:** `one_hot(y, C)` trả về ma trận $(m, C)$, hàng $i$ có số 1 ở cột `y[i]`.

In [ ]:
def one_hot(y, C):
    ### START CODE HERE ### (≈ 2 dòng)
    Y = np.zeros((len(y), C))
    Y[np.arange(len(y)), y] = 1
    ### END CODE HERE ###
    return Y


Y_train = one_hot(y_train, C)

In [ ]:
check_close(one_hot(np.array([2, 0]), 3), [[0, 0, 1], [1, 0, 0]], message="one_hot")
check_shape(Y_train, (len(y_train), 10))

## 2. Softmax ổn định số học

**Bài tập:** `softmax(Z)` theo **hàng**, trừ max của mỗi hàng trước khi `exp`.

In [ ]:
def softmax(Z):
    ### START CODE HERE ### (≈ 2 dòng)
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    A = E / E.sum(axis=1, keepdims=True)
    ### END CODE HERE ###
    return A

In [ ]:
check_close(softmax(np.array([[1.0, 2.0, 3.0]])), [[0.09003057, 0.24472847, 0.66524096]], tol=1e-7)
check_close(softmax(np.array([[1000.0, 1000.0]])), [[0.5, 0.5]], message="không tràn số")

## 3. Cost và gradient

$$A = \text{softmax}(XW + \mathbf b), \quad J = -\frac1m\sum_{i}\sum_k Y_{ik}\log A_{ik} + \frac{\lambda}{2m}\lVert W\rVert^2$$

$$\nabla_W J = \frac1m X^\top(A - Y) + \frac{\lambda}{m}W, \qquad \nabla_{\mathbf b} J = \frac1m\sum_i (A_{i,:} - Y_{i,:})$$

In [ ]:
def cost_and_grads(W, b, X, Y, lam=0.0):
    m = len(X)
    ### START CODE HERE ### (≈ 5 dòng)
    A = softmax(X @ W + b)
    J = -np.sum(Y * np.log(A + 1e-12)) / m + lam / (2 * m) * np.sum(W**2)
    dZ = (A - Y) / m
    dW = X.T @ dZ + lam / m * W
    db = dZ.sum(axis=0)
    ### END CODE HERE ###
    return J, dW, db

In [ ]:
W0, b0 = np.zeros((64, C)), np.zeros(C)
check_close(cost_and_grads(W0, b0, X_train, Y_train)[0], np.log(10), tol=1e-6, message="J(W=0) = ln 10")
Wr = np.random.default_rng(0).normal(0, 0.1, (64, C))
num = numerical_gradient(lambda w: cost_and_grads(w.reshape(64, C), b0, X_train[:30], Y_train[:30], 2.0)[0], Wr.ravel().copy())
check_true(relative_error(cost_and_grads(Wr, b0, X_train[:30], Y_train[:30], 2.0)[1].ravel(), num) < 1e-6, "gradient check dW")

**Đầu ra kỳ vọng:** $J(W = 0) = \ln 10 \approx 2.3026$ — mọi lớp đều có xác suất 0.1.

## 4. Train bằng mini-batch gradient descent

**Bài tập:** mỗi epoch xáo trộn dữ liệu, duyệt từng mini-batch, cập nhật $W, \mathbf b$. Ghi lại cost trên toàn tập train mỗi epoch.

⚠️ **Bẫy hay gặp:** objective là $\frac1M\sum(\text{CE}) + \frac{\lambda}{2M}\lVert W\rVert^2$ với $M$ là số mẫu của **toàn bộ** tập train. Gradient trên một mini-batch phải ước lượng gradient của **objective đầy đủ**, nên số hạng regularization phải là $\frac{\lambda}{M}W$ — không phải $\frac{\lambda}{|\text{batch}|}W$. Vì hàm `cost_and_grads` chia cho số mẫu nó nhận vào, khi gọi trên một batch hãy truyền $\lambda' = \lambda\cdot\frac{|\text{batch}|}{M}$. Quên điều này làm regularization mạnh gấp $M/|\text{batch}|$ lần (≈ 20 lần ở đây) → model underfit nặng.

In [ ]:
def train(X, Y, lr=0.5, epochs=60, batch_size=64, lam=1.0, seed=0):
    rng = np.random.default_rng(seed)
    W, b = np.zeros((X.shape[1], Y.shape[1])), np.zeros(Y.shape[1])
    history = []
    for _ in range(epochs):
        idx = rng.permutation(len(X))
        for s in range(0, len(X), batch_size):
            batch = idx[s : s + batch_size]
            ### START CODE HERE ### (≈ 3 dòng)
            _, dW, db = cost_and_grads(W, b, X[batch], Y[batch], lam * len(batch) / len(X))
            W = W - lr * dW
            b = b - lr * db
            ### END CODE HERE ###
        history.append(cost_and_grads(W, b, X, Y, lam)[0])
    return W, b, history


W, b, history = train(X_train, Y_train)
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.show()

## 5. Đánh giá

**Bài tập:** viết `predict(W, b, X)` trả về lớp có xác suất cao nhất; tính accuracy trên tập validation.

In [ ]:
def predict(W, b, X):
    ### START CODE HERE ### (≈ 1 dòng)
    pred = np.argmax(X @ W + b, axis=1)
    ### END CODE HERE ###
    return pred


val_accuracy = np.mean(predict(W, b, X_val) == y_val)
print(f"Accuracy validation: {val_accuracy:.4f}")

In [ ]:
check_greater(val_accuracy, 0.93, "accuracy validation > 93%")

*(Câu hỏi)* Vì sao `argmax` của **logit** $XW + \mathbf b$ cho cùng kết quả với `argmax` của **xác suất** softmax?

**Đáp án:** softmax là hàm **đồng biến** theo từng logit (trong cùng một hàng, mẫu số giống nhau) → thứ tự các lớp không đổi, nên không cần tính softmax khi chỉ cần nhãn.

Trực quan hoá trọng số $W$ của từng lớp:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for k, ax in enumerate(axes):
    w = W[:, k].reshape(8, 8)
    ax.imshow(w, cmap="RdBu", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
    ax.set_title(str(k))
    ax.axis("off")
plt.show()

## 6. Áp dụng cho Fashion-MNIST *(đề bài AIVN dự kiến)*

**Fashion-MNIST** (Zalando, 2017): 70 000 ảnh xám 28×28 của 10 loại quần áo, giày dép, túi — "người kế nhiệm" khó hơn của bộ chữ số MNIST. Các hàm bạn đã viết ở mục 1–5 dùng lại **nguyên vẹn**, chỉ khác dữ liệu.

> ⏬ Lần chạy đầu tải ~30 MB (một lần, lưu trong `data/raw/fashion-mnist`).

In [ ]:
from ml_course.real_datasets import FASHION_MNIST_CLASSES, load_fashion_mnist

X_fm_train_img, y_fm_train, X_fm_test_img, y_fm_test = load_fashion_mnist()
print("train:", X_fm_train_img.shape, "| test:", X_fm_test_img.shape, "| giá trị pixel:", X_fm_train_img.min(), "→", X_fm_train_img.max())
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for k, ax in enumerate(axes):
    ax.imshow(X_fm_train_img[np.flatnonzero(y_fm_train == k)[0]], cmap="gray_r")
    ax.set_title(FASHION_MNIST_CLASSES[k], fontsize=9)
    ax.axis("off")
plt.show()

**Bài tập 6a:** duỗi mỗi ảnh thành vector 784 chiều và chia 255 → `X_fm_train`, `X_fm_test`; tạo nhãn one-hot `Y_fm_train` bằng hàm `one_hot` của bạn.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
X_fm_train = X_fm_train_img.reshape(len(X_fm_train_img), -1) / 255.0
X_fm_test = X_fm_test_img.reshape(len(X_fm_test_img), -1) / 255.0
Y_fm_train = one_hot(y_fm_train, 10)
### END CODE HERE ###

In [ ]:
check_shape(X_fm_train, (60_000, 784))
check_shape(Y_fm_train, (60_000, 10))
check_true(0 <= X_fm_test.min() and X_fm_test.max() <= 1, "pixel đã đưa về [0, 1]")

**Bài tập 6b:** train bằng hàm `train` của bạn với `lr=0.1, epochs=10, batch_size=64, lam=1.0`, rồi tính `fm_test_accuracy` trên **tập test** bằng `predict`.

> 💡 Vì sao không dùng `lr=0.5` như với `digits`? Với 60 000 mẫu, mỗi epoch có ~940 bước cập nhật (digits chỉ ~21): learning rate lớn làm $W$ dao động mạnh quanh cực tiểu — nhớ lại [chương 06 · bài 04](../../../notebooks/06_linear_regression/04_batch_sgd_minibatch_gd.ipynb).

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
W_fm, b_fm, history_fm = train(X_fm_train, Y_fm_train, lr=0.1, epochs=10, batch_size=64, lam=1.0)
fm_pred = predict(W_fm, b_fm, X_fm_test)
fm_test_accuracy = np.mean(fm_pred == y_fm_test)
### END CODE HERE ###
print(f"Fashion-MNIST — accuracy test: {fm_test_accuracy:.4f}")

In [ ]:
check_greater(fm_test_accuracy, 0.80, "softmax regression trên Fashion-MNIST: accuracy test > 80%")

**Bài tập 6c:** tính accuracy **theo từng lớp** `per_class_acc` (mảng 10 phần tử: tỉ lệ đoán đúng trong các ảnh test thuộc lớp $k$) và tên lớp khó nhất `hardest` (dùng `FASHION_MNIST_CLASSES`).

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
per_class_acc = np.array([np.mean(fm_pred[y_fm_test == k] == k) for k in range(10)])
hardest = FASHION_MNIST_CLASSES[int(np.argmin(per_class_acc))]
### END CODE HERE ###
for name, acc_k in zip(FASHION_MNIST_CLASSES, per_class_acc):
    print(f"{name:>14s}: {acc_k:.2f}")
print("lớp khó nhất:", hardest)

In [ ]:
check_shape(per_class_acc, (10,))
check_true(hardest in ("áo phông", "áo len", "áo khoác", "áo sơ mi"), "lớp khó nhất là một loại áo — các loại áo có hình dáng gần giống nhau")

Trọng số của từng lớp, vẽ thành ảnh 28×28 (đỏ = pixel sáng làm **tăng** điểm của lớp, xanh = làm giảm):

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
lim = np.abs(W_fm).max()
for k, ax in enumerate(axes):
    ax.imshow(W_fm[:, k].reshape(28, 28), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(FASHION_MNIST_CLASSES[k], fontsize=9)
    ax.axis("off")
plt.show()

*(Câu hỏi)* Vì sao accuracy trên Fashion-MNIST (~83%) thấp hơn hẳn trên `digits` (~95%), và vì sao các loại **áo** hay bị nhầm với nhau?

**Đáp án:** mô hình tuyến tính so ảnh với **một ảnh mẫu** cho mỗi lớp (hình trọng số ở trên). Quần, giày, túi có **hình dáng** khác hẳn nhau nên dễ phân biệt; còn áo phông, áo len, áo khoác, áo sơ mi có cùng một bóng dáng thân áo + tay áo — khác biệt nằm ở **chi tiết** (cổ áo, hàng cúc, độ dài tay, hoa văn) mà một ảnh mẫu tuyến tính không thể biểu diễn. Cần đặc trưng phi tuyến: mạng nhiều lớp ([chương 27](../../../notebooks/27_deep_learning_foundations/README.md)) và CNN ([chương 29](../../../notebooks/29_cnn/README.md)) đạt khoảng 90% trở lên trên chính bộ dữ liệu này.

### Tổng kết

- Cài đặt Softmax Regression hoàn chỉnh: one-hot, softmax ổn định, cross-entropy + L2, gradient, mini-batch GD.
- Đây chính là **lớp output** của mọi neural network phân loại — chương 27 sẽ thêm các tầng ẩn phía trước.
- Cùng một code: ~95% trên `digits`, ~83% trên Fashion-MNIST — giới hạn của mô hình tuyến tính lộ rõ khi các lớp chỉ khác nhau ở chi tiết.